# PropTech 360 | Data Engineering Graduation Capstone
### Industry: Commercial real estate and property management · Level: Intermediate · Total: 100 points

**Student name:** ____________________  **Cohort:** _______APRIL________  **Submission date:** ______OCT 10, 2026_______

> **Assessment notebook, not a worked solution.** Execute the dataset generator, complete the `TODO` tasks independently, and provide evidence of local and AWS runs. Use fictional, synthetic records only.

**Scenario.** MetroSpace Property Group manages a portfolio of commercial buildings. Property, unit, lease, repair, and energy records arrive as disconnected CSV exports. Management cannot reliably compare occupancy, earned rental revenue, maintenance SLA performance, or energy use across buildings. You have been hired to design a reproducible analytics pipeline and explain its reliability and operating cost.

**Mission.** Ingest raw source files; detect and address quality defects; transform data at scale; model and query a PostgreSQL mart; then land curated outputs in an AWS serverless analytics architecture. Your solution must be rerunnable without silently doubling records.

## 1. Five assessed topics and end-to-end architecture

1. **Python ingestion and file handling** — load five raw CSV extracts and record ingestion metadata.
2. **Pandas ETL and data quality** — profile, validate, quarantine, and clean datasets with explicit quality rules.
3. **PySpark transformations** — derive monthly occupancy, rental revenue, maintenance SLA, and energy metrics.
4. **PostgreSQL, SQL, and analytical modeling** — implement a dimensional mart and solve business questions.
5. **AWS serverless pipeline** — S3 raw/curated zones, Lambda, Glue Data Catalog, Athena, IAM, and CloudWatch.

**Reference architecture**

`CSV source extracts → Python intake / raw zone → Pandas validation + quarantine → PySpark curated Parquet → PostgreSQL dimensional mart → SQL reporting`

`                                                                     ↘ S3 curated → Glue Catalog → Athena`  
`S3 raw → S3 event → Lambda (validation / orchestration metadata) → S3 audit + CloudWatch`

**Design constraint:** Do not run Spark inside Lambda. Lambda handles lightweight event validation or metadata; perform the Spark workload separately, locally or on an appropriately provisioned service. Do not claim that S3 automatically refreshes PostgreSQL.

## 2. Business requirements and KPI definitions

Report results **per property and month (January–June 2026)** and provide portfolio-wide totals where appropriate.

- **Occupancy rate:** occupied units / total active units at the **month end**. A unit is occupied when at least one valid lease has `start_date <= month_end <= end_date` (or `end_date` is missing). Flag conflicting simultaneously active leases for the same unit instead of counting the unit twice.
- **Earned monthly rent (simplified):** sum of contractual `monthly_rent_usd` for leases active on the **last day of that month**. This is an analytical proxy, **not cash collected, prorated rent, or recognized accounting revenue**.
- **Maintenance SLA compliance:** share of resolved work orders whose `response_hours` is within target: CRITICAL ≤ 4 h, HIGH ≤ 12 h, MEDIUM ≤ 48 h, LOW ≤ 72 h. Exclude unresolved orders from the compliance denominator and report their count separately. Define and document any additional exclusions.
- **Monthly energy intensity:** sum of valid `energy_kwh` / total property floor area in m²; state whether your chosen denominator includes all or occupied units. Never divide by zero.

**Stakeholders:** portfolio director, operations manager, finance analyst, and cloud platform team. **Grain discipline:** do not join work-order line items directly to lease line items when computing rent; this can multiply totals.

## 3. Environment and setup

**Local:** Python 3.12 recommended for compatibility; pandas, numpy, pyspark, pyarrow, sqlalchemy, psycopg (or psycopg2), Jupyter; PostgreSQL; Java matching your installed PySpark version. Use VS Code or Jupyter. Match PySpark and Java versions to their official compatibility guidance. A local PostgreSQL database and AWS account with appropriate permissions are required for their respective implementation tasks.

**AWS:** S3, Lambda, Glue Data Catalog (crawler or explicit table definitions), Athena, IAM, and CloudWatch. Use a dedicated learning account or permitted lab. AWS resources can incur charges; set a budget alert, avoid sensitive data, restrict IAM privileges, and delete resources after assessment. **Never embed access keys or passwords in notebook outputs or Git.**

**Suggested repository structure**
```text
proptech360-capstone/
  capstone.ipynb
  requirements.txt
  README.md
  data/raw/              # five generated source files
  data/quarantine/       # rejected records + reasons
  data/curated/          # partitioned Parquet + audit/summary
  src/                   # reusable ingestion, validation and ETL modules
  sql/                   # DDL, upserts, reporting queries
  aws/                   # Lambda handler + IAM policy description + deployment steps
  tests/                 # checks for quality rules and KPI logic
  evidence/              # redacted run screenshots / Athena results
```

**Installation (run in the notebook's Python environment):** `python -m pip install pandas numpy pyspark pyarrow sqlalchemy psycopg[binary] pytest boto3` (or pin compatible versions in `requirements.txt`). The next cell checks common local imports; install packages before running it.

In [ ]:
# Environment check: record the versions in your README.
import sys
import importlib.metadata as metadata
print('Python:', sys.version.split()[0])
for package in ['pandas', 'numpy', 'pyspark', 'pyarrow', 'sqlalchemy', 'psycopg', 'boto3']:
    try:
        print(f'{package}: {metadata.version(package)}')
    except metadata.PackageNotFoundError:
        print(f'{package}: NOT INSTALLED (install before its assessed section)')

## 4. Generate the synthetic source extracts

Run the generator **once or rerun with the same seed**; it overwrites the same five source files deterministically. Dates and properties are fictional. Deliberately injected quality issues include repeated work orders, negative meter readings, invalid foreign keys, missing response durations, and duplicate unit identifiers. **Do not edit raw files manually**; preserve them as immutable source inputs and fix issues in your pipeline.

**Source contract**

| File | Grain | Primary identifier | Important fields |
|---|---|---|---|
| `properties.csv` | one property | `property_id` | name, city |
| `units.csv` | one unit | `unit_id` | property_id, floor_area_sqm, active |
| `leases.csv` | one lease | `lease_id` | unit_id, start/end dates, monthly_rent_usd |
| `work_orders.csv` | one work order | `work_order_id` | unit_id, opened/closed dates, priority, response_hours, cost_usd |
| `meter_readings.csv` | unit-month | `reading_id` | unit_id, reading_month, energy_kwh |

The file formats intentionally contain dates as text; you must parse and validate them. The generator is **test fixture setup**, not a graded solution.

In [ ]:
from pathlib import Path
import csv
import random
from datetime import date, timedelta

rng = random.Random(3602026)
root = Path('data')
raw = root / 'raw'
raw.mkdir(parents=True, exist_ok=True)
for folder in ['quarantine', 'curated']:
    (root / folder).mkdir(parents=True, exist_ok=True)

def write_csv(name, rows):
    path = raw / name
    with path.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    print(f'{path}: {len(rows):,} rows')

cities = ['Lagos', 'Abuja', 'Ibadan', 'Port Harcourt']
properties = [dict(property_id=f'P{i:03}', property_name=f'MetroSpace Tower {i:02}',
                   city=cities[(i-1) % len(cities)]) for i in range(1, 13)]
units = []
for p in properties:
    for j in range(1, 21):
        units.append(dict(unit_id=f"{p['property_id']}-U{j:03}", property_id=p['property_id'],
                          floor_area_sqm=rng.choice([45, 60, 75, 90, 120, 150]), active=1))
leases = []
for i, unit in enumerate(units, start=1):
    if rng.random() < 0.22:  # intentionally vacant unit
        continue
    start = date(2025, rng.randint(1, 12), rng.randint(1, 25))
    end = date(2026, rng.randint(3, 12), rng.randint(1, 28)) if rng.random() < 0.27 else None
    leases.append(dict(lease_id=f'L{i:05}', unit_id=unit['unit_id'],
                       start_date=start.isoformat(), end_date=end.isoformat() if end else '',
                       monthly_rent_usd=rng.choice([650, 800, 975, 1200, 1500, 1800, 2200])))
priorities = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
work_orders = []
for i in range(1, 1501):
    unit = rng.choice(units)
    opened = date(2026, rng.randint(1, 6), rng.randint(1, 28))
    resolved = rng.random() < 0.82
    response = round(rng.uniform(1, 96), 1)
    work_orders.append(dict(work_order_id=f'WO{i:06}', unit_id=unit['unit_id'],
        opened_date=opened.isoformat(), closed_date=(opened + timedelta(days=rng.randint(0, 15))).isoformat() if resolved else '',
        priority=rng.choice(priorities), response_hours=response if resolved else '',
        cost_usd=round(rng.uniform(20, 1250), 2), status='RESOLVED' if resolved else 'OPEN'))
months = [f'2026-{m:02}-01' for m in range(1, 7)]
readings = []
for unit in units:
    for month in months:
        readings.append(dict(reading_id=f'R{len(readings)+1:06}', unit_id=unit['unit_id'],
                             reading_month=month, energy_kwh=round(rng.uniform(100, 1250), 2)))
# Known intentionally dirty records; do not remove these from the raw source.
work_orders.append(work_orders[8].copy())                     # exact duplicate
work_orders[34]['unit_id'] = 'P999-U999'                       # unknown parent
work_orders[52]['response_hours'] = ''                         # resolved with missing response
work_orders[63]['cost_usd'] = '-300'                           # invalid negative repair cost
readings[17]['energy_kwh'] = '-45'                             # invalid negative consumption
readings[34]['unit_id'] = 'P999-U999'                           # unknown unit
units.append(units[4].copy())                                 # duplicate identifier
for name, rows in [('properties.csv', properties), ('units.csv', units), ('leases.csv', leases),
                   ('work_orders.csv', work_orders), ('meter_readings.csv', readings)]:
    write_csv(name, rows)
print('Raw fixture ready. Retain a copy before any transformations.')

## 5. Topic 1 — Python ingestion and reproducibility (15 points)

**Tasks:** (a) create a reusable loader for the five source files; (b) validate file existence, expected columns, and nonempty inputs; (c) record filename, row count, ingest timestamp UTC, and SHA-256 digest in an audit manifest; (d) make repeated runs safe—raw source files must remain unchanged; (e) expose clean error messages for a missing file or missing required column.

**Evidence:** reusable `src/ingest.py`, audit manifest, and a test showing controlled failure for a nonexistent path. Use `pathlib`, `hashlib`, `datetime`, `csv` and/or pandas. **Do not** use `print()` as your only error handling strategy.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

RAW_DIR = Path("data/raw")
CURATED_DIR = Path("data/curated")

EXPECTED_COLUMNS = {
    "properties.csv": ["property_id", "property_name", "city"],
    "units.csv": ["unit_id", "property_id", "floor_area_sqm", "active"],
    "leases.csv": ["lease_id", "unit_id", "start_date", "end_date", "monthly_rent_usd"],
    "work_orders.csv": [
        "work_order_id",
        "unit_id",
        "opened_date",
        "closed_date",
        "priority",
        "response_hours",
        "cost_usd",
        "status",
    ],
    "meter_readings.csv": ["reading_id", "unit_id", "reading_month", "energy_kwh"],
}

datasets = {}
ingestion_manifest = []

for filename, cols in EXPECTED_COLUMNS.items():
    filepath = RAW_DIR / filename

    if not filepath.exists():
        raise FileNotFoundError(f"Source file not found: {filepath}")

    try:
        df = pd.read_csv(
            filepath, dtype=str, keep_default_na=False, encoding="utf-8-sig"
        )
    except Exception as e:
        raise ValueError(f"{filename}: empty file or invalid CSV.") from e

    missing = set(cols) - set(df.columns)
    if missing:
        raise ValueError(
            f"{filename}: missing required columns: {', '.join(sorted(missing))}"
        )
    if df.empty:
        raise ValueError(f"{filename}: no data rows found.")

    datasets[filename] = df
    ingestion_manifest.append(
        {
            "filename": filename,
            "row_count": len(df),
            "ingest_timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "sha256": hashlib.sha256(filepath.read_bytes()).hexdigest(),
        }
    )

# Save manifest
CURATED_DIR.mkdir(parents=True, exist_ok=True)
with open(CURATED_DIR / "ingestion_manifest.json", "w", encoding="utf-8") as f:
    json.dump(ingestion_manifest, f, indent=2)
    f.write("\n")

# Access variables individually and print row counts
properties = datasets["properties.csv"]
units = datasets["units.csv"]
leases = datasets["leases.csv"]
work_orders = datasets["work_orders.csv"]
meter_readings = datasets["meter_readings.csv"]

for name, df in datasets.items():
    print(f"{name}: {len(df)}")

## 6. Topic 2 — Pandas validation, cleaning, and quarantine (20 points)

**Tasks:** Profile null counts, duplicate identifiers, date parse failures, invalid category values, non-positive areas, negative money/energy, broken foreign keys, invalid chronological order, and overlapping leases. Implement a **documented policy** for each rule: fail the run, quarantine the row, correct deterministically with justification, or retain and flag. Preserve original raw files. Do not silently drop data or fill unknown business values with zero.

**Required validations:** unique property/unit/lease/work-order/reading IDs; valid `property_id`→properties and `unit_id`→units; start ≤ end for dated leases; opened ≤ closed for resolved work orders; `priority` ∈ {CRITICAL,HIGH,MEDIUM,LOW}; valid status; numeric nonnegative `energy_kwh`, `cost_usd`, and `monthly_rent_usd`; positive `floor_area_sqm`; `reading_month` must fall inside Jan–Jun 2026; at most one reading per `(unit_id, reading_month)`.

**Quarantine contract:** produce a file per dataset containing original row, `reason_code`, and `run_id`. Report input, accepted, quarantined, and duplicate counts, with an explicit reconciliation explaining whether exact duplicates are counted as quarantined. For aggregate KPI computation, a work order missing an SLA response can be retained for volume reporting but must not count as SLA-compliant by default. If you choose another defensible treatment, document it.

**Evidence:** `src/quality.py`, `data/quarantine/`, reconciliation CSV/JSON, at least five assertions or unit tests including foreign-key and duplicate checks.

In [ ]:
# STUDENT TASK 2: Pandas validation, cleaning, and quarantine.
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
import json


# TODO: return row count, nulls by column, duplicate identifier count, and types.
def profile_table(df: pd.DataFrame, identifier: str) -> dict:
    nulls = {}
    for col in df.columns:
        # Check for NaN, empty strings, or stringified whitespace
        empty_count = 0
        for val in df[col]:
            if pd.isna(val) or str(val).strip() == "":
                empty_count += 1
        nulls[col] = empty_count

    return {
        "rows": len(df),
        "nulls": nulls,
        "duplicate_identifiers": int(df.duplicated(identifier).sum()),
        "types": df.dtypes.astype(str).to_dict(),
    }


# TODO: parse dates and numbers, enforce source and FK rules, reconcile rejects.
def validate_and_quarantine(datasets: dict, run_id: str):
    id_columns = {
        "properties.csv": "property_id",
        "units.csv": "unit_id",
        "leases.csv": "lease_id",
        "work_orders.csv": "work_order_id",
        "meter_readings.csv": "reading_id",
    }
    sla_hours = {"CRITICAL": 4, "HIGH": 12, "MEDIUM": 48, "LOW": 72}
    clean, quarantine, summary = {}, {}, {}

    for name, id_col in id_columns.items():
        original = datasets[name].copy().reset_index(drop=True)
        df = original.copy()

        # Clean string whitespace
        for col in df.columns:
            df[col] = df[col].fillna("").astype(str).str.strip()

        # Convert known dates
        date_cols = [
            "start_date",
            "end_date",
            "opened_date",
            "closed_date",
            "reading_month",
        ]
        for col in date_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], format="%Y-%m-%d", errors="coerce")

        # Convert known numbers
        num_cols = [
            "floor_area_sqm",
            "monthly_rent_usd",
            "cost_usd",
            "energy_kwh",
            "response_hours",
        ]
        for col in num_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        problems = {i: [] for i in range(len(df))}

        # Duplicate ID and Exact Duplicate checks
        exact = original.duplicated()
        duplicates = df.loc[~exact].duplicated(id_col, keep=False)
        for i in df.index:
            if exact[i]:
                problems[i].append("EXACT_DUPLICATE")
            if duplicates.get(i, False):
                problems[i].append("DUPLICATE_ID")

        # Row-by-row validation checks
        for i, row in df.iterrows():
            errors = []

            # 1. Primary ID
            if pd.isna(row.get(id_col)) or str(row.get(id_col)).strip() == "":
                errors.append("MISSING_ID")

            # Mandatory dates
            for col in ["start_date", "opened_date", "reading_month"]:
                if col in row and pd.isna(row[col]):
                    errors.append("INVALID_" + col.upper())

            # Mandatory / Valid numbers
            for col in num_cols:
                if col in row:
                    if col == "response_hours" and pd.isna(row[col]):
                        continue
                    if pd.isna(row[col]) or row[col] < 0 or row[col] == float("inf"):
                        errors.append("INVALID_" + col.upper())

            # 2. Foreign Key checks
            if name == "units.csv":
                valid_props = list(clean["properties.csv"]["property_id"])
                if row["property_id"] not in valid_props:
                    errors.append("UNKNOWN_PROPERTY")
            elif name != "properties.csv" and "unit_id" in row:
                valid_units = list(clean["units.csv"]["unit_id"])
                if row["unit_id"] not in valid_units:
                    errors.append("UNKNOWN_UNIT")

            # 3. File-specific checks
            if name == "properties.csv":
                if not row["property_name"]:
                    errors.append("MISSING_PROPERTY_NAME")
                if not row["city"]:
                    errors.append("MISSING_CITY")

            elif name == "units.csv":
                if row["floor_area_sqm"] == 0:
                    errors.append("ZERO_AREA")
                if str(row["active"]) not in ["0", "1"]:
                    errors.append("INVALID_ACTIVE")

            elif name == "leases.csv":
                if pd.notna(row["start_date"]) and pd.notna(row["end_date"]):
                    if row["start_date"] > row["end_date"]:
                        errors.append("LEASE_DATE_ORDER")

            elif name == "work_orders.csv":
                if row["priority"] not in sla_hours:
                    errors.append("INVALID_PRIORITY")
                if row["status"] not in ["OPEN", "RESOLVED"]:
                    errors.append("INVALID_STATUS")
                if row["status"] == "RESOLVED" and pd.isna(row["closed_date"]):
                    errors.append("MISSING_CLOSED_DATE")
                if row["status"] == "OPEN" and pd.notna(row["closed_date"]):
                    errors.append("OPEN_WITH_CLOSED_DATE")
                if pd.notna(row["opened_date"]) and pd.notna(row["closed_date"]):
                    if row["opened_date"] > row["closed_date"]:
                        errors.append("WORK_ORDER_DATE_ORDER")

            elif name == "meter_readings.csv":
                start, end = pd.Timestamp("2026-01-01"), pd.Timestamp("2026-06-30")
                if pd.isna(row["reading_month"]) or not (
                    start <= row["reading_month"] <= end
                ):
                    errors.append("READING_OUT_OF_RANGE")

            problems[i].extend(errors)

            # Re-check lost inputs from non-required fields
            for col in ["end_date", "closed_date", "response_hours"]:
                if col in df:
                    value = original.loc[i, col]
                    if (
                        pd.notna(value)
                        and str(value).strip() != ""
                        and pd.isna(row[col])
                    ):
                        problems[i].append("INVALID_" + col.upper())

        # Additional multi-row validations
        if name == "meter_readings.csv":
            readings = df.loc[~exact].copy()
            readings["reading_month"] = readings["reading_month"].dt.strftime("%Y-%m")
            repeated = readings.duplicated(["unit_id", "reading_month"], keep=False)
            for i in readings.index[repeated]:
                problems[i].append("DUPLICATE_UNIT_MONTH")

        if name == "leases.csv":
            valid_indices = [i for i in df.index if len(problems[i]) == 0]
            valid = df.loc[valid_indices].copy()
            valid["end_date"] = valid["end_date"].fillna(pd.Timestamp.max)
            for _, leases in valid.groupby("unit_id"):
                for i, a in leases.iterrows():
                    for j, b in leases.iterrows():
                        if (
                            i < j
                            and a["start_date"] <= b["end_date"]
                            and b["start_date"] <= a["end_date"]
                        ):
                            problems[i].append("OVERLAPPING_LEASE")
                            problems[j].append("OVERLAPPING_LEASE")

        if name == "work_orders.csv":
            df["missing_response"] = df["response_hours"].isna()
            limit = df["priority"].map(sla_hours)
            df["sla_compliant"] = (
                (df["status"] == "RESOLVED") & (df["response_hours"] <= limit)
            ).fillna(False)

        # Separate clean vs rejected
        rejected_mask = [len(problems[i]) > 0 for i in range(len(df))]

        clean_df = df[[not r for r in rejected_mask]].copy()
        quarantine_df = original[rejected_mask].copy()

        # Add tracking metadata
        reasons = [
            ";".join(sorted(set(problems[i])))
            for i in range(len(df))
            if rejected_mask[i]
        ]
        quarantine_df["reason_code"] = reasons
        quarantine_df["run_id"] = run_id

        clean[name] = clean_df
        quarantine[name] = quarantine_df

        # Aggregate summary metrics
        rule_counts = {}
        for errors in problems.values():
            for reason in set(errors):
                rule_counts[reason] = rule_counts.get(reason, 0) + 1

        if name == "work_orders.csv":
            rule_counts["MISSING_RESPONSE_RETAINED"] = int(
                clean_df["missing_response"].sum()
            )

        summary[name] = {
            **profile_table(original, id_col),
            "exact_duplicates": int(exact.sum()),
            "rule_counts": rule_counts,
            "run_id": run_id,
            "input": len(df),
            "accepted": len(clean_df),
            "quarantined": len(quarantine_df),
        }

    return clean, quarantine, summary


# TODO: tests for duplicate work order, unknown unit, negative kWh, missing response and bad date.
test_data = {
    "properties.csv": pd.DataFrame(
        [{"property_id": "P1", "property_name": "Tower", "city": "Lagos"}]
    ),
    "units.csv": pd.DataFrame(
        [{"unit_id": "U1", "property_id": "P1", "floor_area_sqm": "50", "active": "1"}]
    ),
    "leases.csv": pd.DataFrame(
        [
            {
                "lease_id": "L1",
                "unit_id": "U1",
                "start_date": "bad-date",
                "end_date": "",
                "monthly_rent_usd": "500",
            }
        ]
    ),
    "work_orders.csv": pd.DataFrame(
        [
            {
                "work_order_id": "W1",
                "unit_id": "U1",
                "opened_date": "2026-01-01",
                "closed_date": "2026-01-02",
                "priority": "HIGH",
                "response_hours": "",
                "cost_usd": "20",
                "status": "RESOLVED",
            }
        ]
    ),
    "meter_readings.csv": pd.DataFrame(
        [
            {
                "reading_id": "R1",
                "unit_id": "U1",
                "reading_month": "2026-01-01",
                "energy_kwh": "-10",
            },
            {
                "reading_id": "R2",
                "unit_id": "UNKNOWN",
                "reading_month": "2026-02-01",
                "energy_kwh": "100",
            },
        ]
    ),
}

# Add exact duplicate work order
test_data["work_orders.csv"] = pd.concat(
    [test_data["work_orders.csv"], test_data["work_orders.csv"]], ignore_index=True
)

test_clean, test_rejected, test_summary = validate_and_quarantine(test_data, "test-run")

assert "EXACT_DUPLICATE" in test_rejected["work_orders.csv"].iloc[0]["reason_code"]
assert "UNKNOWN_UNIT" in test_rejected["meter_readings.csv"].iloc[1]["reason_code"]
assert (
    "INVALID_ENERGY_KWH" in test_rejected["meter_readings.csv"].iloc[0]["reason_code"]
)
assert "INVALID_START_DATE" in test_rejected["leases.csv"].iloc[0]["reason_code"]
assert test_clean["work_orders.csv"].iloc[0]["missing_response"]
assert not test_clean["work_orders.csv"].iloc[0]["sla_compliant"]

for counts in test_summary.values():
    assert counts["input"] == counts["accepted"] + counts["quarantined"]

print("Topic 2 example checks passed.")


# TODO: persist clean records into a staging area, separate from untouched raw data.
run_id = datetime.now(timezone.utc).isoformat()
clean_datasets, quarantine_datasets, quality_summary = validate_and_quarantine(
    datasets, run_id
)

for folder, tables in [
    ("staging", clean_datasets),
    ("quarantine", quarantine_datasets),
]:
    output = Path("data") / folder
    output.mkdir(parents=True, exist_ok=True)
    for filename, table in tables.items():
        table.to_csv(output / filename, index=False)

for counts in quality_summary.values():
    assert counts["input"] == counts["accepted"] + counts["quarantined"]

report = Path("data/curated/quality_summary.json")
report.parent.mkdir(parents=True, exist_ok=True)
report.write_text(json.dumps(quality_summary, indent=2), encoding="utf-8")

pd.DataFrame(quality_summary).T[["input", "accepted", "quarantined"]]

## 7. Topic 3 — PySpark curated transformations (25 points)

**Tasks:** (a) start a SparkSession; load validated staging data using explicit schemas where practical; (b) generate the six calendar month ends; (c) compute a unit-month occupancy and monthly-rent snapshot without duplicate lease multiplication; (d) aggregate maintenance per property-month with compliant and noncompliant **resolved** cases plus open cases; (e) aggregate valid energy by property-month; (f) join the independently aggregated KPI tables on `(property_id, month)`; (g) write partitioned Parquet by `month` to `data/curated/property_month_kpis/`.

**Critical transformations:** use DataFrame joins, groupBy/aggregations, casts, date operations, `when`, and window functions where appropriate. Avoid collecting entire fact tables to the driver. Verify all 12 properties × 6 months = **72 property-month output rows** provided no property has been intentionally excluded. Use `countDistinct` or a pre-aggregated unit-month snapshot to avoid multiplying occupancy and rent during joins.

**Minimum columns:** `property_id`, `month`, `total_units`, `occupied_units`, `occupancy_rate`, `earned_monthly_rent_usd`, `resolved_orders`, `sla_eligible_orders`, `sla_compliant_orders`, `open_orders`, `sla_compliance_rate`, `energy_kwh`, `floor_area_sqm`, `energy_intensity_kwh_sqm`.

**Edge cases:** no leases at month end, all orders unresolved, missing response durations, zero/invalid floor area, and duplicate identifiers. Explain your chosen denominator for each rate. Write a quality assertion for unique `(property_id, month)` and a range check for percentages.

In [ ]:
import os
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. SETUP SPARK SESSION
project_folder = (
    Path.cwd().parent if Path.cwd().name == "capstone.ipynb" else Path.cwd()
)
postgres_driver = project_folder / "postgresql-42.7.13.jar"

if not postgres_driver.exists():
    raise FileNotFoundError(f"Driver not found: {postgres_driver}")

spark = (
    SparkSession.builder.appName("PropTech360-Capstone")
    .config("spark.jars", str(postgres_driver))
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)


# 2. READ STAGING DATASETS WITH EXPLICIT SCHEMAS
properties_schema = "property_id STRING, property_name STRING, city STRING"
units_schema = "unit_id STRING, property_id STRING, floor_area_sqm DOUBLE, active INT"
leases_schema = "lease_id STRING, unit_id STRING, start_date DATE, end_date DATE, monthly_rent_usd DOUBLE"
work_orders_schema = "work_order_id STRING, unit_id STRING, opened_date DATE, closed_date DATE, priority STRING, response_hours DOUBLE, cost_usd DOUBLE, status STRING, missing_response BOOLEAN, sla_compliant BOOLEAN"
meter_readings_schema = (
    "reading_id STRING, unit_id STRING, reading_month DATE, energy_kwh DOUBLE"
)

data_dir = project_folder / "data" / "staging"

properties_df = (
    spark.read.option("header", True)
    .option("mode", "FAILFAST")
    .schema(properties_schema)
    .csv(str(data_dir / "properties.csv"))
)
units_df = (
    spark.read.option("header", True)
    .option("mode", "FAILFAST")
    .schema(units_schema)
    .csv(str(data_dir / "units.csv"))
)
leases_df = (
    spark.read.option("header", True)
    .option("mode", "FAILFAST")
    .schema(leases_schema)
    .csv(str(data_dir / "leases.csv"))
)
work_orders_df = (
    spark.read.option("header", True)
    .option("mode", "FAILFAST")
    .schema(work_orders_schema)
    .csv(str(data_dir / "work_orders.csv"))
)
meter_readings_df = (
    spark.read.option("header", True)
    .option("mode", "FAILFAST")
    .schema(meter_readings_schema)
    .csv(str(data_dir / "meter_readings.csv"))
)


# 3. VALIDATE ID COLUMNS (MISSING & DUPLICATE CHECKS)
tables_to_check = [
    (properties_df, "property_id", "properties"),
    (units_df, "unit_id", "units"),
    (leases_df, "lease_id", "leases"),
    (work_orders_df, "work_order_id", "work_orders"),
    (meter_readings_df, "reading_id", "meter_readings"),
]

for df, id_col, table_name in tables_to_check:
    # Check nulls or empty strings
    missing_count = df.filter(
        F.col(id_col).isNull() | (F.trim(F.col(id_col)) == "")
    ).count()
    if missing_count > 0:
        raise ValueError(f"Missing ID found in table: {table_name}")

    # Check duplicates
    duplicate_count = df.groupBy(id_col).count().filter("count > 1").count()
    if duplicate_count > 0:
        raise ValueError(f"Duplicate ID found in table: {table_name}")


# 4. CREATE REPORTING MONTHS CALENDAR
dates_data = [
    ("2026-01-01",),
    ("2026-02-01",),
    ("2026-03-01",),
    ("2026-04-01",),
    ("2026-05-01",),
    ("2026-06-01",),
]

calendar_df = spark.createDataFrame(dates_data, ["month"])
calendar_df = calendar_df.withColumn("month", F.to_date(F.col("month")))
calendar_df = calendar_df.withColumn("month_end", F.last_day(F.col("month")))


# 5. BUILD UNIT-MONTH LEASE SNAPSHOT
# Filter active units and build unit calendar matrix
active_units_df = units_df.filter(F.col("active") == 1)
unit_months_df = active_units_df.crossJoin(calendar_df)

# Join leases to active unit months
joined_leases_df = unit_months_df.join(leases_df, on="unit_id", how="left")

# Filter for leases active on the month-end date
active_lease_condition = (F.col("start_date") <= F.col("month_end")) & (
    F.col("end_date").isNull() | (F.col("end_date") >= F.col("month_end"))
)
current_leases_df = joined_leases_df.filter(active_lease_condition)

# Validate no overlapping active leases per unit-month
overlap_count = (
    current_leases_df.groupBy("unit_id", "month").count().filter("count > 1").count()
)
if overlap_count > 0:
    raise ValueError("Overlapping leases at month end; rerun Topic 2.")

# Select active lease details
current_leases_df = current_leases_df.select(
    "unit_id", "month", "monthly_rent_usd"
).withColumn("occupied", F.lit(1))

# Join back to create the full unit-month snapshot
snapshot_df = unit_months_df.join(
    current_leases_df, on=["unit_id", "month"], how="left"
)
snapshot_df = snapshot_df.fillna({"occupied": 0, "monthly_rent_usd": 0.0})


# 6. COMPUTE METRICS SEPARATELY
# A. Occupancy & Rent KPIs
occupancy_kpi_df = snapshot_df.groupBy("property_id", "month").agg(
    F.count("*").alias("total_units"),
    F.sum("occupied").alias("occupied_units"),
    F.sum("monthly_rent_usd").alias("earned_monthly_rent_usd"),
)

# B. Work Order Maintenance KPIs
units_prop_lookup = units_df.select("unit_id", "property_id")
orders_df = work_orders_df.join(units_prop_lookup, on="unit_id", how="left")
orders_df = orders_df.withColumn("month", F.trunc(F.col("opened_date"), "month"))

# Define SLA response limits by priority
sla_limit = (
    F.when(F.col("priority") == "CRITICAL", 4)
    .when(F.col("priority") == "HIGH", 12)
    .when(F.col("priority") == "MEDIUM", 48)
    .when(F.col("priority") == "LOW", 72)
)

is_resolved = F.col("status") == "RESOLVED"
is_compliant = (
    is_resolved
    & (F.col("response_hours") >= 0)
    & (F.col("response_hours") <= sla_limit)
)
is_missing_resp = is_resolved & F.col("response_hours").isNull()
is_open = F.col("status") == "OPEN"

maintenance_kpi_df = orders_df.groupBy("property_id", "month").agg(
    F.sum(F.when(is_resolved, 1).otherwise(0)).alias("resolved_orders"),
    F.sum(F.when(is_compliant, 1).otherwise(0)).alias("sla_compliant_orders"),
    F.sum(F.when(is_missing_resp, 1).otherwise(0)).alias("missing_response_orders"),
    F.sum(F.when(is_open, 1).otherwise(0)).alias("open_orders"),
)

maintenance_kpi_df = maintenance_kpi_df.withColumn(
    "sla_eligible_orders", F.col("resolved_orders")
)
maintenance_kpi_df = maintenance_kpi_df.withColumn(
    "sla_noncompliant_orders", F.col("resolved_orders") - F.col("sla_compliant_orders")
)

# C. Energy & Area Metrics
readings_df = meter_readings_df.join(units_prop_lookup, on="unit_id", how="left")
readings_df = readings_df.withColumn("month", F.trunc(F.col("reading_month"), "month"))

energy_kpi_df = readings_df.groupBy("property_id", "month").agg(
    F.sum("energy_kwh").alias("energy_kwh")
)

property_area_df = units_df.groupBy("property_id").agg(
    F.sum("floor_area_sqm").alias("floor_area_sqm")
)


# 7. COMBINE ALL KPIS INTO FINAL PROPERTY-MONTH TABLE
base_spine_df = properties_df.select("property_id").crossJoin(
    calendar_df.select("month")
)

final_kpis_df = base_spine_df.join(
    occupancy_kpi_df, on=["property_id", "month"], how="left"
)
final_kpis_df = final_kpis_df.join(
    maintenance_kpi_df, on=["property_id", "month"], how="left"
)
final_kpis_df = final_kpis_df.join(
    energy_kpi_df, on=["property_id", "month"], how="left"
)
final_kpis_df = final_kpis_df.join(property_area_df, on="property_id", how="left")

# Fill missing metric values with zero
fill_zero_cols = [
    "total_units",
    "occupied_units",
    "earned_monthly_rent_usd",
    "resolved_orders",
    "sla_eligible_orders",
    "sla_compliant_orders",
    "sla_noncompliant_orders",
    "missing_response_orders",
    "open_orders",
    "energy_kwh",
]
final_kpis_df = final_kpis_df.fillna(0, subset=fill_zero_cols)

# Compute rate columns using safe division
final_kpis_df = final_kpis_df.withColumn(
    "occupancy_rate",
    F.when(
        F.col("total_units") > 0, F.col("occupied_units") / F.col("total_units")
    ).otherwise(None),
)

final_kpis_df = final_kpis_df.withColumn(
    "sla_compliance_rate",
    F.when(
        F.col("sla_eligible_orders") > 0,
        F.col("sla_compliant_orders") / F.col("sla_eligible_orders"),
    ).otherwise(None),
)

final_kpis_df = final_kpis_df.withColumn(
    "energy_intensity_kwh_sqm",
    F.when(
        F.col("floor_area_sqm") > 0, F.col("energy_kwh") / F.col("floor_area_sqm")
    ).otherwise(None),
)


# 8. DATA QUALITY ASSERTIOS & DISPLAY
if final_kpis_df.count() != 72:
    raise ValueError("Unexpected property-month row count. Expected 72.")

unique_keys_count = final_kpis_df.select("property_id", "month").distinct().count()
if unique_keys_count != 72:
    raise ValueError("Duplicate property-month keys found.")

invalid_occupancy = final_kpis_df.filter(
    (F.col("occupancy_rate") < 0) | (F.col("occupancy_rate") > 1)
).count()
if invalid_occupancy > 0:
    raise ValueError("Invalid occupancy_rate found outside range [0, 1].")

invalid_sla = final_kpis_df.filter(
    (F.col("sla_compliance_rate") < 0) | (F.col("sla_compliance_rate") > 1)
).count()
if invalid_sla > 0:
    raise ValueError("Invalid sla_compliance_rate found outside range [0, 1].")

final_kpis_df.orderBy("property_id", "month").show(6, truncate=False)


# 9. WRITE CURATED OUTPUT TO PARQUET
output_path = project_folder / "data" / "curated" / "property_month_kpis"

(final_kpis_df.write.mode("overwrite").partitionBy("month").parquet(str(output_path)))

## 8. Topic 4 — PostgreSQL dimensional mart and analytics (20 points)

**Tasks:** Create a PostgreSQL database `proptech360` and dimensional schema with `dim_property`, `dim_month`, and `fact_property_month`. Document natural keys, surrogate keys if used, primary and foreign keys, NOT NULL constraints, valid-range checks, and indexes. Load the curated property-month dataset into the fact table using parameterized inserts or a bulk loader. Implement an idempotent `INSERT ... ON CONFLICT ... DO UPDATE` strategy at the `(property_id, month)` grain; rerun loading and prove row counts stay stable.

**Write and submit SQL answering:**

1. Rank properties by average monthly occupancy for January–June 2026, displaying numerator and denominator context.
2. Calculate monthly earned contractual rent by city (not payment receipts).
3. Show maintenance SLA compliance by property, including counts of eligible resolved and excluded unresolved/missing-response records.
4. Find property-months with energy intensity above the portfolio's monthly average; show both intensity and total kWh.
5. Create a portfolio dashboard view with occupancy rate, earned rent, SLA compliance, and energy intensity; explain handling of NULL rates.

**Evidence:** `sql/01_schema.sql`, `02_load.sql` or `src/load_postgres.py`, `03_analytics.sql`, and screenshots/text output for five queries. Verify PostgreSQL results reconcile with Spark aggregates. Do not paste database credentials into the notebook.

In [ ]:
import os
import psycopg2
import psycopg2.extras
from psycopg2 import sql
import pandas as pd
from pathlib import Path
from dotenv import dotenv_values
from pyspark.sql import functions as F

# 1. CONFIGURATION & ENVIRONMENT
settings = dotenv_values(project_folder / '.env', interpolate=False)
settings.update(os.environ)

required_vars = ['DB_NAME', 'DB_USER', 'DB_PASSWORD', 'DB_HOST', 'DB_PORT']
for var in required_vars:
    assert settings.get(var), f"Set {var} in .env file."

jdbc_url = f"jdbc:postgresql://{settings['DB_HOST']}:{settings['DB_PORT']}/{settings['DB_NAME']}"
jdbc_options = {
    'user': settings['DB_USER'],
    'password': settings['DB_PASSWORD'],
    'driver': 'org.postgresql.Driver'
}

# 2. SQL SCHEMAS AND QUERIES 
schema_sql = """
CREATE SCHEMA IF NOT EXISTS mart;

CREATE TABLE IF NOT EXISTS mart.dim_property (
    property_id TEXT PRIMARY KEY CHECK (property_id != ''),
    property_name TEXT NOT NULL CHECK (property_name != ''),
    city TEXT NOT NULL CHECK (city != '')
);

CREATE TABLE IF NOT EXISTS mart.dim_month (
    month DATE PRIMARY KEY CHECK (EXTRACT(DAY FROM month) = 1)
);

CREATE TABLE IF NOT EXISTS mart.fact_property_month (
    property_id TEXT REFERENCES mart.dim_property(property_id) NOT NULL,
    month DATE REFERENCES mart.dim_month(month) NOT NULL,
    total_units INTEGER NOT NULL CHECK (total_units >= 0),
    occupied_units INTEGER NOT NULL CHECK (occupied_units BETWEEN 0 AND total_units),
    earned_monthly_rent_usd NUMERIC(18,2) NOT NULL CHECK (earned_monthly_rent_usd >= 0),
    resolved_orders INTEGER NOT NULL CHECK (resolved_orders >= 0),
    sla_eligible_orders INTEGER NOT NULL CHECK (sla_eligible_orders = resolved_orders),
    sla_compliant_orders INTEGER NOT NULL CHECK (sla_compliant_orders BETWEEN 0 AND sla_eligible_orders),
    sla_noncompliant_orders INTEGER NOT NULL CHECK (sla_noncompliant_orders = resolved_orders - sla_compliant_orders),
    missing_response_orders INTEGER NOT NULL CHECK (missing_response_orders BETWEEN 0 AND sla_noncompliant_orders),
    open_orders INTEGER NOT NULL CHECK (open_orders >= 0),
    energy_kwh NUMERIC(18,2) NOT NULL CHECK (energy_kwh >= 0),
    floor_area_sqm NUMERIC(18,2) CHECK (floor_area_sqm >= 0),
    occupancy_rate NUMERIC GENERATED ALWAYS AS 
        (CAST(occupied_units AS NUMERIC) / NULLIF(total_units, 0)) STORED,
    sla_compliance_rate NUMERIC GENERATED ALWAYS AS 
        (CAST(sla_compliant_orders AS NUMERIC) / NULLIF(sla_eligible_orders, 0)) STORED,
    energy_intensity_kwh_sqm NUMERIC GENERATED ALWAYS AS 
        (energy_kwh / NULLIF(floor_area_sqm, 0)) STORED,
    PRIMARY KEY (property_id, month)
);

CREATE INDEX IF NOT EXISTS fact_month_idx ON mart.fact_property_month(month);
CREATE INDEX IF NOT EXISTS property_city_idx ON mart.dim_property(city);

CREATE OR REPLACE VIEW mart.portfolio_dashboard AS
SELECT month,
       SUM(total_units) AS total_units,
       SUM(occupied_units) AS occupied_units,
       CAST(SUM(occupied_units) AS NUMERIC) / NULLIF(SUM(total_units), 0) AS occupancy_rate,
       SUM(earned_monthly_rent_usd) AS earned_monthly_rent_usd,
       SUM(sla_eligible_orders) AS sla_eligible_orders,
       SUM(sla_compliant_orders) AS sla_compliant_orders,
       SUM(open_orders) AS open_orders,
       CAST(SUM(sla_compliant_orders) AS NUMERIC) / NULLIF(SUM(sla_eligible_orders), 0) AS sla_compliance_rate,
       SUM(energy_kwh) AS energy_kwh,
       SUM(floor_area_sqm) AS floor_area_sqm,
       SUM(energy_kwh) / NULLIF(SUM(floor_area_sqm), 0) AS energy_intensity_kwh_sqm
FROM mart.fact_property_month
GROUP BY month;
"""

property_insert = """
INSERT INTO mart.dim_property (property_id, property_name, city) VALUES (%s, %s, %s)
ON CONFLICT (property_id) DO UPDATE SET
    property_name = EXCLUDED.property_name, city = EXCLUDED.city
"""

month_insert = """
INSERT INTO mart.dim_month (month) VALUES (%s) ON CONFLICT (month) DO NOTHING
"""

fact_columns = [
    'property_id', 'month', 'total_units', 'occupied_units', 'earned_monthly_rent_usd',
    'resolved_orders', 'sla_eligible_orders', 'sla_compliant_orders', 'sla_noncompliant_orders',
    'missing_response_orders', 'open_orders', 'energy_kwh', 'floor_area_sqm',
]

stage_sql = """
CREATE TEMP TABLE stage_property_month ON COMMIT DROP AS
SELECT property_id, month, total_units, occupied_units, earned_monthly_rent_usd,
       resolved_orders, sla_eligible_orders, sla_compliant_orders, sla_noncompliant_orders,
       missing_response_orders, open_orders, energy_kwh, floor_area_sqm
FROM mart.fact_property_month WITH NO DATA
"""

stage_insert = """INSERT INTO stage_property_month VALUES 
    (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)"""

upsert_sql = """
INSERT INTO mart.fact_property_month (
    property_id, month, total_units, occupied_units, earned_monthly_rent_usd,
    resolved_orders, sla_eligible_orders, sla_compliant_orders, sla_noncompliant_orders,
    missing_response_orders, open_orders, energy_kwh, floor_area_sqm
)
SELECT property_id, month, total_units, occupied_units, earned_monthly_rent_usd,
       resolved_orders, sla_eligible_orders, sla_compliant_orders, sla_noncompliant_orders,
       missing_response_orders, open_orders, energy_kwh, floor_area_sqm
FROM stage_property_month
ON CONFLICT (property_id, month) DO UPDATE SET
    total_units = EXCLUDED.total_units,
    occupied_units = EXCLUDED.occupied_units,
    earned_monthly_rent_usd = EXCLUDED.earned_monthly_rent_usd,
    resolved_orders = EXCLUDED.resolved_orders,
    sla_eligible_orders = EXCLUDED.sla_eligible_orders,
    sla_compliant_orders = EXCLUDED.sla_compliant_orders,
    sla_noncompliant_orders = EXCLUDED.sla_noncompliant_orders,
    missing_response_orders = EXCLUDED.missing_response_orders,
    open_orders = EXCLUDED.open_orders,
    energy_kwh = EXCLUDED.energy_kwh,
    floor_area_sqm = EXCLUDED.floor_area_sqm;
"""

# 3. DATA PREPARATION 
# Read the validated file directly; another cell may have reused "tables".
properties = spark.read.csv(
    str(project_folder / 'data/staging/properties.csv'),
    header=True,
    schema='property_id STRING, property_name STRING, city STRING',
    mode='FAILFAST'
)
facts = spark.read.parquet(str(project_folder / 'data/curated/property_month_kpis'))
months = facts.select('month').distinct()

assert facts.count() == 72
assert facts.select('property_id', 'month').distinct().count() == 72

# Extract PySpark DataFrame data into standard Python lists of tuples for loading
props_data = [tuple(row) for row in properties.collect()]
months_data = [(row['month'],) for row in months.collect()]
facts_data = [tuple(row) for row in facts.select(*fact_columns).collect()]

# 4. DATABASE INITIALIZATION & LOADING 
# Connect to postgres first because the project database may not exist yet.
admin_connection = psycopg2.connect(
    host=settings['DB_HOST'], port=settings['DB_PORT'], dbname='postgres',
    user=settings['DB_USER'], password=settings['DB_PASSWORD']
)
try:
    admin_connection.autocommit = True
    with admin_connection.cursor() as admin_cursor:
        admin_cursor.execute(
            'SELECT 1 FROM pg_database WHERE datname = %s', (settings['DB_NAME'],)
        )
        if admin_cursor.fetchone() is None:
            admin_cursor.execute(
                sql.SQL('CREATE DATABASE {}').format(sql.Identifier(settings['DB_NAME']))
            )
finally:
    admin_connection.close()

conn = psycopg2.connect(
    host=settings['DB_HOST'],
    port=settings['DB_PORT'],
    dbname=settings['DB_NAME'],
    user=settings['DB_USER'],
    password=settings['DB_PASSWORD']
)
conn.autocommit = False 
cursor = conn.cursor()

try:
    cursor.execute(schema_sql)
    cursor.execute(stage_sql)
    
    # Batch execute the inserts using standard python drivers
    psycopg2.extras.execute_batch(cursor, property_insert, props_data)
    psycopg2.extras.execute_batch(cursor, month_insert, months_data)
    psycopg2.extras.execute_batch(cursor, stage_insert, facts_data)

    counts = []
    for run_number in range(2):
        cursor.execute(upsert_sql)
        cursor.execute("SELECT COUNT(*) FROM mart.fact_property_month")
        counts.append(cursor.fetchone()[0])

    assert counts == [72, 72], 'Expected 72 rows after both loads.'
    conn.commit()
    print('Both loads have 72 fact rows.')
except Exception as e:
    conn.rollback()
    raise e
finally:
    cursor.close()
    conn.close()

# 5. VALIDATION 
constraint_sql = """
SELECT table_name, constraint_name, constraint_type
FROM information_schema.table_constraints WHERE table_schema = 'mart'
"""
constraints = spark.read.jdbc(jdbc_url, f"({constraint_sql}) AS checks", properties=jdbc_options)
constraints.orderBy('table_name', 'constraint_name').show(100, truncate=False)

stored = spark.read.jdbc(jdbc_url, 'mart.fact_property_month', properties=jdbc_options)
expected = facts.orderBy('property_id', 'month').toPandas()
actual = stored.select(*facts.columns).orderBy('property_id', 'month').toPandas()

for column in facts.columns:
    if column != 'property_id' and column != 'month':
        expected[column] = pd.to_numeric(expected[column])
        actual[column] = pd.to_numeric(actual[column])

pd.testing.assert_frame_equal(actual, expected, check_dtype=False, check_exact=False, rtol=1e-9, atol=1e-7)
print('All 72 rows reconcile with Spark.')

# 6. ANALYTICS & EXPORT 
queries = [
    """
    SELECT p.property_id, p.property_name,
           DENSE_RANK() OVER (ORDER BY AVG(f.occupancy_rate) DESC NULLS LAST) AS occupancy_rank,
           AVG(f.occupancy_rate) AS average_monthly_occupancy,
           SUM(f.occupied_units) AS occupied_unit_months,
           SUM(f.total_units) AS available_unit_months,
           COUNT(f.occupancy_rate) AS months_with_defined_rate
    FROM mart.fact_property_month f JOIN mart.dim_property p USING (property_id)
    WHERE month BETWEEN DATE '2026-01-01' AND DATE '2026-06-01'
    GROUP BY p.property_id, p.property_name
    ORDER BY occupancy_rank, p.property_id
    """,
    """
    SELECT f.month, p.city, SUM(f.earned_monthly_rent_usd) AS earned_monthly_rent_usd
    FROM mart.fact_property_month f JOIN mart.dim_property p USING (property_id)
    WHERE month BETWEEN DATE '2026-01-01' AND DATE '2026-06-01'
    GROUP BY f.month, p.city
    ORDER BY f.month, p.city
    """,
    """
    SELECT property_id, SUM(resolved_orders) AS resolved_orders,
           SUM(sla_eligible_orders) AS eligible_resolved_orders,
           SUM(sla_compliant_orders) AS compliant_orders,
           SUM(sla_noncompliant_orders) AS noncompliant_orders,
           SUM(open_orders) AS excluded_unresolved_orders,
           SUM(missing_response_orders) AS included_missing_response_orders,
           0 AS excluded_missing_response_orders,
           CAST(SUM(sla_compliant_orders) AS NUMERIC) / NULLIF(SUM(sla_eligible_orders), 0) AS sla_compliance_rate
    FROM mart.fact_property_month
    WHERE month BETWEEN DATE '2026-01-01' AND DATE '2026-06-01'
    GROUP BY property_id
    ORDER BY property_id
    """,
    """
    SELECT f.property_id, f.month, f.energy_kwh, f.energy_intensity_kwh_sqm,
           p.energy_intensity_kwh_sqm AS portfolio_energy_intensity
    FROM mart.fact_property_month f JOIN mart.portfolio_dashboard p USING (month)
    WHERE f.energy_intensity_kwh_sqm > p.energy_intensity_kwh_sqm
      AND f.month BETWEEN DATE '2026-01-01' AND DATE '2026-06-01'
    ORDER BY f.month, f.property_id
    """,
    """
    SELECT * FROM mart.portfolio_dashboard
    WHERE month BETWEEN DATE '2026-01-01' AND DATE '2026-06-01'
    ORDER BY month
    """
]

evidence_dir = project_folder / 'evidence' / '04_postgres'
evidence_dir.mkdir(parents=True, exist_ok=True)

constraints.toPandas().to_csv(evidence_dir / 'constraints.csv', index=False)

for idx, q_sql in enumerate(queries, start=1):
    res_df = spark.read.jdbc(jdbc_url, f"({q_sql}) AS result", properties=jdbc_options)
    res_df.show(10, truncate=False)
    res_df.toPandas().to_csv(evidence_dir / f'query_{idx}.csv', index=False)

## 9. Topic 5 — AWS serverless lake and Athena (15 points)

**Create and document the following pipeline in your permitted AWS environment:**

1. Configure a uniquely named **private S3 bucket** with `raw/`, `curated/`, and `audit/` prefixes. Upload a raw CSV to `raw/`; upload **your actual locally generated curated Parquet** to `curated/property_month_kpis/` (preserving partitions). Avoid public ACLs; use encryption defaults and least-privilege IAM.
2. Create a **Python Lambda** triggered by new objects under `raw/`. The handler checks prefix, extension, nonzero size and relevant object metadata, then records an audit JSON under `audit/` and emits structured logs to **CloudWatch**. Add robust handling for event retries and duplicate notifications (e.g., deterministic audit key based on bucket/key/version or eTag and idempotent overwrite), and **prevent a trigger loop** by only triggering on `raw/`.
3. Register the curated Parquet dataset with the **Glue Data Catalog**, through a crawler or explicit schema, and validate inferred partition columns and types. Configure an **Athena query result S3 location** with permissions, and query the catalogued dataset.
4. Run Athena queries for top five property-month occupancy values and portfolio monthly energy totals. Reconcile the monthly energy totals against your local Spark/PostgreSQL results. Capture CloudWatch Lambda logs and an Athena query result as evidence.
5. Explain IAM permissions, S3 event notification behavior, region consistency, expected costs, operational failure modes, and teardown steps. **Do not** give Lambda write permissions across all S3 buckets.

**Division of responsibility:** Lambda validates new raw arrivals and writes an audit; the **Spark computation is performed separately** and its outputs are explicitly uploaded to curated storage. There is no claim that the Lambda event by itself performs the full local Spark pipeline.

**If a course-lab account blocks resource creation:** submit `aws/lambda_handler.py`, architecture diagram, least-privilege IAM pseudopolicy with resource scoping, AWS CLI/console deployment steps, sample S3 event and local Lambda test output, Glue/Athena DDL or crawler configuration, and a clear list of steps you could not execute. Your instructor should apply the same preannounced evidence policy across the cohort.

In [ ]:
# STUDENT TASK 5: Minimal AWS Lambda handler outline — save the completed version to aws/lambda_handler.py.
# This handler is NOT run automatically by the notebook. Deploy it separately in Lambda.
LAMBDA_HANDLER_STARTER = r'''
import json
import os
import hashlib
import boto3
from urllib.parse import unquote_plus

s3 = boto3.client('s3')
AUDIT_BUCKET = os.environ.get('AUDIT_BUCKET')


def lambda_handler(event, context):
    processed = 0

    # TODO: iterate Records safely; validate s3:ObjectCreated event, raw/ prefix and .csv suffix.
    for record in event.get('Records', []):
        if record.get('eventSource') != 'aws:s3':
            continue
        if not record.get('eventName', '').startswith('ObjectCreated:'):
            continue

        bucket = record['s3']['bucket']['name']
        key = unquote_plus(record['s3']['object']['key'])
        if not key.startswith('raw/') or not key.lower().endswith('.csv'):
            continue  # Ignore audit files so they cannot trigger a loop.

        # TODO: decode the object key, inspect metadata with head_object and reject zero-size files.
        # The key was decoded above so names containing spaces work correctly.
        version = record['s3']['object'].get('versionId')
        request = {'Bucket': bucket, 'Key': key}
        if version:
            request['VersionId'] = version
        metadata = s3.head_object(**request)
        if metadata['ContentLength'] == 0:
            raise ValueError(f'Empty CSV file: {key}')

        etag = metadata['ETag'].strip('"')
        event_etag = record['s3']['object'].get('eTag', etag).strip('"')
        if not version and event_etag != etag:
            print(json.dumps({'status': 'superseded', 'key': key}))
            continue  # A newer upload has replaced this unversioned file.

        # TODO: derive idempotent audit key from bucket/key/object version or eTag.
        # Repeated notifications use the same filename instead of adding duplicates.
        identity = json.dumps([bucket, key, version or etag])
        audit_key = 'audit/' + hashlib.sha256(identity.encode()).hexdigest() + '.json'

        # TODO: write JSON audit under audit/ with scoped S3 permissions; structured logs.
        audit = {
            'source_bucket': bucket,
            'source_file': key,
            'size_bytes': metadata['ContentLength'],
            'content_type': metadata.get('ContentType'),
            'version_id': version,
            'etag': etag,
            'status': 'accepted'
        }
        s3.put_object(
            Bucket=AUDIT_BUCKET or bucket,
            Key=audit_key,
            Body=json.dumps(audit),
            ContentType='application/json',
            ServerSideEncryption='AES256'
        )
        print(json.dumps(audit))
        processed += 1

    # TODO: handle or report partial failure without swallowing exceptions.
    # S3 errors raise an exception automatically, allowing Lambda to retry.
    # Audits already written are safely overwritten during that retry.
    return {'statusCode': 200, 'files_audited': processed}
'''
print('Lambda code ready. Copy the code inside LAMBDA_HANDLER_STARTER into Lambda.')


## 10. Testing, deliverables, and presentation (5 points)

**Automated validation checklist:** prove the generator can be rerun; raw hashes stay unchanged; key uniqueness and foreign keys hold after cleaning; quarantine reasons reconcile; Spark emits 72 unique property-month rows; occupancy and SLA rates are between 0 and 1 or NULL when denominator is zero; monthly rent is not multiplied by work-order counts; PostgreSQL repeated upsert retains 72 fact rows; Athena energy totals reconcile with local output for a selected month.

**Submit:** completed `.ipynb`; runnable repository with `README.md`, pinned `requirements.txt`, `src/`, `sql/`, `aws/`, `tests/`, generated synthetic raw data, sample quarantine/curated outputs (or reproducible generation instructions), a short architecture diagram, and redacted evidence. In README, document commands, assumptions, quality decisions, AWS resource names **without credentials**, rerun instructions, and cleanup. Provide a **5–8 minute walkthrough**: the business problem, one intentional data defect, local KPI reconciliation, AWS evidence, and one improvement you would make for production.

**Suggested assessment window:** 10–14 days, individual work or clearly attributed group contributions. Code can be modularized outside the notebook; the notebook should remain the readable end-to-end assessment entry point.

## 11. Grading rubric (100 points)

| Topic | Points | Assessor looks for |
|---|---:|---|
| Python ingestion | 15 | Reusable loader (5); schema/error validation (4); audit digest/metadata (4); rerun safety (2) |
| Pandas data quality | 20 | Profiling and rule coverage (8); explicit quarantine + reasons (6); reconciliation (3); tests (3) |
| PySpark ETL | 25 | Correct grain and snapshot logic (8); SLA and energy transformations (7); robust joins, edge cases and assertions (6); partitioned curated output (4) |
| PostgreSQL and SQL | 20 | Schema and constraints (5); idempotent load (5); five accurate business queries (8); Spark/SQL reconciliation (2) |
| AWS serverless | 15 | Scoped S3 and Lambda event/audit (6); Glue/Athena results (5); IAM/CloudWatch, cost, failure handling and teardown (4) |
| Testing, documentation and demo | 5 | Repeatable tests (2); README/architecture (2); clear demo (1) |
| **Total** | **100** | |

**Assessment principles:** Award points for correct, reproducible and evidenced work, not merely screenshots or copied service configurations. A correct alternative design is acceptable when it meets the stated grain, definitions, validation, and security requirements. Record your institution's policy for restricted AWS lab access before grading.

### Final student reflection

Write 200–300 words: Which source defect most threatened KPI correctness? Why is the property-month fact grain useful? What is one drawback of the serverless design? What would you improve for production scale, monitoring, or data governance?